# 한국 배당성장주 진성 블라인드(백지상태) OOS 검증 연구실 (Google Colab)

**목적:** 전체 기간(`2017~2026`) 결과를 미리 알고 전략을 고를 때 생기는 **사후 선별 편향(Hindsight / Data Snooping Bias)**의 실제 크기를 측정하고, **백지상태(Blind)에서 과거 구간만 보고 자동 선정한 전략이 한 번도 보지 않은 미래 구간에서도 살아남는지** 엄격히 검증합니다.

### 엄격한 시간 봉인(Strict Hold-Out) 설계
| 구간 | 사업연도 | 실투자 구간 | 역할 |
|---|---|---|---|
| **탐색(학습) 구간 (Train)** | `FY2016 ~ FY2020` | `2017-04 ~ 2022-04` (5.0년) | Colab이 **이 구간 데이터만 보고** 10개 후보 팩터 중 상위 팩터·가중치를 백지상태에서 자동 선정 후 **영구 동결** |
| **봉인 검증 구간 (Blind OOS)** | `FY2021 ~ FY2025` | `2022-04 ~ 최신` (4.7년) | `2022-04-01` 이후 정보 0% 개입 상태에서 **오직 채점만 수행** |

### 무엇을 비교·측정하는가?
1. **백지 자동 선정 3종 (`2017~2022`만 보고 동결):**
   - `Blind-1 (자동 Top-3 균등)`: 탐색 구간 ICIR 상위 3개 팩터를 자동 선택해 `1/3` 균등 적용
   - `Blind-2 (자동 IC-수축 앙상블)`: 탐색 구간 양(+)의 IC 팩터를 50% 수축 가중
   - `Blind-3 (학습 0% 단순 균등)`: 4대 보편 팩터(저변동·절대밸류·주주환원·퀄리티)를 학습 없이 `25%` 균등 적용
2. **사후 편향 거품(Hindsight Gap) 정량 측정:**
   - 우리가 전체 기간을 보고 만든 `사후 최종안 ②`의 OOS 성과와 `Blind-1/2`의 OOS 성과를 나란히 비교해 **실제 사후 편향 거품이 몇 %p인지** 투명하게 공개합니다.
3. **OOS 4중 견고성 스트레스 테스트 (`2022-04 ~ 최신`):**
   - ① **OOS 무작위 500회 몬테카를로(`p-value`)**
   - ② **종목 수 민감도 (`TOP_N = 8, 10, 15, 20`)**
   - ③ **리밸런싱 시점 이동 (`4월 초` vs `5월 초` vs `10월 초`)**
   - ④ **OOS 월별 적립식 + 매도규칙 장기보유 (매월 100만 원, FAIL/2연속삭감 외 매도 금지)**

> **API 키 불필요:** Google Drive `Colab Notebooks/dividend_alpha_lab/`의 기존 1,252 PIT DB와 3,000일 시세 캐시(`ckpt_prices_3000d_...`)를 자동으로 공유해 초고속으로 실행됩니다.


In [ ]:
# ==============================================================================
# 1. 환경 설정 · 한글 폰트(fonts-nanum) 및 PDF 도구 자동 설치 (API 키 불필요)
# ==============================================================================
import os, sys, re, io, gzip, json, math, time, base64, random, warnings, statistics
from pathlib import Path
from datetime import datetime, timezone, timedelta
from html import escape
from concurrent.futures import ThreadPoolExecutor, as_completed
import subprocess

warnings.filterwarnings("ignore")
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("[환경] Google Colab 감지 -> 한글 폰트(fonts-nanum) 및 PDF 엔진 설치 중...")
    subprocess.run(["apt-get", "-qq", "update"], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    subprocess.run(["apt-get", "-qq", "install", "-y", "fonts-nanum", "libpango-1.0-0", "libpangoft2-1.0-0"], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "weasyprint"], check=False)

import numpy as np
import pandas as pd
import requests
import matplotlib
if not IN_COLAB:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib.ticker as mtick

try:
    from IPython.display import display
except ImportError:
    display = print

def setup_korean_font():
    chosen = None
    for p in [
        "/usr/share/fonts/truetype/nanum/NanumGothic.ttf",
        "/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf",
    ]:
        if os.path.exists(p):
            fm.fontManager.addfont(p)
            if chosen is None:
                chosen = fm.FontProperties(fname=p).get_name()
    if chosen is None:
        avail = {f.name for f in fm.fontManager.ttflist}
        for cand in ["NanumGothic", "AppleGothic", "Malgun Gothic", "DejaVu Sans"]:
            if cand in avail:
                chosen = cand
                break
    try:
        plt.style.use("seaborn-v0_8-whitegrid")
    except Exception:
        pass
    if chosen:
        plt.rcParams["font.family"] = chosen
    plt.rcParams["axes.unicode_minus"] = False
    plt.rcParams["figure.dpi"] = 120
    return chosen

FONT_NAME = setup_korean_font()
print(f"[완료] 한글 폰트: {FONT_NAME} | Python {sys.version.split()[0]} | Colab={IN_COLAB}")

In [ ]:
# ==============================================================================
# 2. 블라인드 OOS 검증 설정 · Google Drive 연결 · 캐시 공유
# ==============================================================================
BLIND_CONFIG = {
    # 엄격한 탐색(Train) / 봉인(OOS) 분할선
    "START_FY": 2016,             # 전체 시작: FY2016 (2017-04 출발)
    "TRAIN_END_FY": 2020,         # 탐색 구간: FY2016~FY2020 (2017-04 ~ 2022-04, 5개 연도, 2022-04-01 직전까지만 학습!)
    "OOS_START_FY": 2021,         # 봉인 검증: FY2021~FY2025 (2022-04 ~ 최신, 완전 미열람 미래 구간!)
    "END_FY": 2025,

    # 포트폴리오 기본 파라미터
    "TOP_N": 10,
    "PIT_MIN_MCAP_EOK": 3000.0,
    "MIN_LIQ_EOK": 10.0,
    "VAL_CAP": 0.50,
    "MAX_FIN_RATIO": 0.50,
    "INITIAL_CAPITAL": 100_000_000.0,
    "MONTHLY_DCA_KRW": 1_000_000.0,
    "FEE_RATE": 0.0025,
    "DIV_TAX_RATE": 0.154,
    "KOSPI_EST_DIV_YIELD": 0.018,

    # 스트레스 테스트 설정
    "MC_RUNS": 500,
    "MC_SEED": 2026,
    "SENS_TOP_NS": [8, 10, 15, 20],
    "SHIFT_MONTHS": [4, 5, 10],   # 4월 초(기준) vs 5월 초 vs 10월 초 리밸런싱 이동 테스트

    # Google Drive 공유 경로 (colab_final_portfolio_backtest와 3,000일 시세 캐시 공유!)
    "USE_GOOGLE_DRIVE": True,
    "DRIVE_DIR": "/content/drive/MyDrive/Colab Notebooks/dividend_alpha_lab",
    "FULL_PIT_FILENAME": "full_kr_fundamentals_pit.jsonl.gz",
    "FORCE_REFRESH_PRICES": False,
    "GITHUB_REPO": "khs0221/Common_Everything",
    "GITHUB_BRANCHES": ["feat/dividend-backtest", "main", "copilot/ai-coding-curriculum"],
    "DATA_SUBPATH": "dividend-backtest/data",
    "CACHE_DIR": "./_cache_blind_oos",
    "WORKERS": 10,
}

CACHE_DIR = Path(BLIND_CONFIG["CACHE_DIR"])
CACHE_DIR.mkdir(parents=True, exist_ok=True)

DRIVE_ROOT = None
DRIVE_REPORT_DIR = None
if IN_COLAB and BLIND_CONFIG["USE_GOOGLE_DRIVE"]:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        DRIVE_ROOT = Path(BLIND_CONFIG["DRIVE_DIR"])
        DRIVE_REPORT_DIR = DRIVE_ROOT / "reports"
        DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
        DRIVE_REPORT_DIR.mkdir(parents=True, exist_ok=True)
        print(f"[Drive 연결 완료] 경로: {DRIVE_ROOT}")
    except Exception as e:
        print(f"[Drive 미연결] 로컬 저장소 사용 ({type(e).__name__}: {e})")

def save_ckpt_json(filename: str, obj: dict) -> None:
    raw = gzip.compress(json.dumps(obj, ensure_ascii=False, separators=(",", ":")).encode("utf-8"), compresslevel=4)
    (CACHE_DIR / filename).write_bytes(raw)
    if DRIVE_ROOT is not None:
        try:
            tmp = DRIVE_ROOT / (filename + ".tmp")
            dst = DRIVE_ROOT / filename
            tmp.write_bytes(raw)
            tmp.replace(dst)
        except Exception:
            pass

def load_ckpt_json(filename: str, force: bool = False) -> dict:
    if force:
        return {}
    search_dirs = [p for p in [DRIVE_ROOT, Path("/tmp/review/nb_test/_cache_final_bt"), CACHE_DIR] if p is not None]
    for base in search_dirs:
        p = base / filename
        if p.exists() and p.stat().st_size > 0:
            try:
                return json.loads(gzip.decompress(p.read_bytes()).decode("utf-8"))
            except Exception:
                continue
    return {}

print(f"[설정 완료] 탐색(Train): FY{BLIND_CONFIG['START_FY']}~FY{BLIND_CONFIG['TRAIN_END_FY']} (2017-04~2022-04) | "
      f"봉인(OOS): FY{BLIND_CONFIG['OOS_START_FY']}~FY{BLIND_CONFIG['END_FY']} (2022-04~최신) | MC={BLIND_CONFIG['MC_RUNS']}회")

In [ ]:
# ==============================================================================
# 3. 1,252개사 PIT DB + 3,000일 수정주가 로드 (기존 캐시 자동 공유)
# ==============================================================================
def load_public_data_bytes(filename: str) -> bytes:
    for p in [Path("../data") / filename, Path("dividend-backtest/data") / filename, Path("/tmp/review") / filename, CACHE_DIR / filename]:
        if p.exists():
            return p.read_bytes()
    for br in BLIND_CONFIG["GITHUB_BRANCHES"]:
        url = f"https://raw.githubusercontent.com/{BLIND_CONFIG['GITHUB_REPO']}/{br}/{BLIND_CONFIG['DATA_SUBPATH']}/{filename}"
        try:
            r = requests.get(url, timeout=30)
            if r.status_code == 200 and len(r.content) > 100:
                (CACHE_DIR / filename).write_bytes(r.content)
                return r.content
        except Exception:
            continue
    raise RuntimeError(f"공개 데이터({filename})를 찾을 수 없습니다.")

def _convert_374_to_pit(rec: dict) -> dict:
    if "pit" in rec:
        return rec
    pit_out = {}
    for fy_k, cols in (rec.get("snapshots") or {}).items():
        if not isinstance(cols, dict):
            continue
        by_y = {}
        for col_name, y_map in cols.items():
            if not isinstance(y_map, dict):
                continue
            for yk, val in y_map.items():
                if val is not None:
                    by_y.setdefault(str(yk), {})[col_name] = val
        pit_out[str(fy_k)] = by_y
    return {
        "ticker": rec["ticker"], "name": rec["name"], "market": rec.get("market", "KOSPI"),
        "path": rec.get("path", "general"), "act_dps": rec.get("act_dps") or {}, "pit": pit_out,
    }

base374_raw = gzip.decompress(load_public_data_bytes("kr_fundamentals_pit.jsonl.gz")).decode("utf-8")
BASE374_MAP = {r["ticker"]: _convert_374_to_pit(r) for r in (json.loads(l) for l in base374_raw.splitlines() if l.strip())}

full_pit_candidates = [
    (DRIVE_ROOT / BLIND_CONFIG["FULL_PIT_FILENAME"]) if DRIVE_ROOT is not None else None,
    Path("/tmp/review") / BLIND_CONFIG["FULL_PIT_FILENAME"],
    CACHE_DIR / BLIND_CONFIG["FULL_PIT_FILENAME"],
]
full_pit_path = next((p for p in full_pit_candidates if p is not None and p.exists() and p.stat().st_size > 1000), None)

if full_pit_path is not None:
    pit_raw = gzip.decompress(full_pit_path.read_bytes()).decode("utf-8")
    full_list = [json.loads(line) for line in pit_raw.splitlines() if line.strip()]
    for rec in full_list:
        b = BASE374_MAP.get(rec["ticker"])
        if not b:
            continue
        for yk, dv in (b.get("act_dps") or {}).items():
            if dv is not None:
                rec.setdefault("act_dps", {})[str(yk)] = dv
        for fy_k, b_frame in (b.get("pit") or {}).items():
            t_frame = rec.setdefault("pit", {}).setdefault(str(fy_k), {})
            for yk, b_cols in b_frame.items():
                t_cols = t_frame.setdefault(str(yk), {})
                for col_k, col_v in b_cols.items():
                    if t_cols.get(col_k) is None and col_v is not None:
                        t_cols[col_k] = col_v
    seen_tks = {r["ticker"] for r in full_list}
    for tk, b in BASE374_MAP.items():
        if tk not in seen_tks:
            full_list.append(b)
    COMPANIES = full_list
    DATASET_LABEL = f"풀 유니버스 PIT DB ({len(COMPANIES):,}개사)"
else:
    COMPANIES = list(BASE374_MAP.values())
    DATASET_LABEL = f"기본 PIT 표본 ({len(COMPANIES):,}개사)"

rf_df = pd.read_csv(io.StringIO(load_public_data_bytes("kr_treasury_10y_annual.csv").decode("utf-8")))
RF_MAP = {int(r["연도"]): float(r["국고채10년_연평균_퍼센트"]) / 100.0 for _, r in rf_df.iterrows()}

_FCHART_URL = "https://fchart.stock.naver.com/sise.nhn"
_ITEM_RE = re.compile(r'<item data="([^"]+)"')
_HEADERS = {"User-Agent": "Mozilla/5.0 (dividend-blind-oos)"}

def fetch_naver_ohlcv_3000(symbol: str, count: int = 3000, min_ymd: str = "20140901") -> list[list]:
    for attempt in range(3):
        try:
            r = requests.get(_FCHART_URL, params={"symbol": symbol, "timeframe": "day", "count": count, "requestType": 0}, headers=_HEADERS, timeout=20)
            r.raise_for_status()
            out = []
            for m in _ITEM_RE.finditer(r.content.decode("euc-kr", errors="ignore")):
                parts = m.group(1).split("|")
                c, v = float(parts[4]), float(parts[5])
                if c > 0 and parts[0] >= min_ymd:
                    out.append([parts[0], c, v])
            if out:
                return out
        except Exception:
            time.sleep(0.35 * (attempt + 1))
    return []

KST = timezone(timedelta(hours=9))
today_tag = datetime.now(KST).strftime("%Y-%m-%d")
CKPT_PX_NAME = f"ckpt_prices_3000d_{today_tag[:7]}_{len(COMPANIES)}.json.gz"
px_ckpt = load_ckpt_json(CKPT_PX_NAME, force=BLIND_CONFIG["FORCE_REFRESH_PRICES"])
px_store = px_ckpt.get("data", {})

all_symbols = list(dict.fromkeys(["KOSPI"] + [c["ticker"] for c in COMPANIES]))
todo_syms = [s for s in all_symbols if s not in px_store]
if px_store and not todo_syms:
    print(f"[캐시 즉시 복원] {CKPT_PX_NAME} ({len(px_store):,}개 종목 3,000일 일봉)")
else:
    print(f"[시세 수집] 보유 {len(px_store):,} -> 남은 {len(todo_syms):,}개 수집 중...")
    t0 = time.time()
    try:
        with ThreadPoolExecutor(max_workers=BLIND_CONFIG["WORKERS"]) as ex:
            futs = {ex.submit(fetch_naver_ohlcv_3000, s): s for s in todo_syms}
            for idx, fut in enumerate(as_completed(futs), 1):
                sym = futs[fut]
                rows = fut.result()
                if rows:
                    px_store[sym] = rows
                if idx % 60 == 0 or idx == len(futs):
                    save_ckpt_json(CKPT_PX_NAME, {"updated": today_tag, "data": px_store})
                    if idx % 240 == 0 or idx == len(futs):
                        print(f"  시세 진행 {len(px_store):,}/{len(all_symbols):,} ({time.time()-t0:.0f}초)")
    finally:
        if todo_syms:
            save_ckpt_json(CKPT_PX_NAME, {"updated": today_tag, "data": px_store})

def _to_df(rows: list[list]) -> pd.DataFrame:
    df = pd.DataFrame(rows, columns=["date", "close", "volume"])
    df["date"] = pd.to_datetime(df["date"], format="%Y%m%d")
    return df.set_index("date").sort_index()

KOSPI_DF = _to_df(px_store["KOSPI"])
PRICE_MAP = {t: _to_df(rows) for t, rows in px_store.items() if t != "KOSPI" and rows}
CALENDAR = KOSPI_DF.index
print(f"[데이터 준비 완료] {DATASET_LABEL} | {len(PRICE_MAP):,}개사 일봉 ({CALENDAR[0].date()} ~ {CALENDAR[-1].date()})")

In [ ]:
# ==============================================================================
# 4. 편향 없는 10개 후보 팩터 패널 구축 (리밸런싱 월 변경 지원: 4월/5월/10월)
# ==============================================================================
CANDIDATE_FACTORS = [
    ("f_lowvol",    "저변동성 (-ann_vol)"),
    ("f_inv_per",   "절대 저PER (-PER)"),
    ("f_inv_pbr",   "절대 저PBR (-PBR)"),
    ("f_yield",     "배당수익률 (FY)"),
    ("f_sh_yield",  "진성 주주환원수익률 (/시총)"),
    ("f_val_rel",   "상대 밸류할인 (±50%캡)"),
    ("f_dps_g",     "정기 DPS 5년 성장률"),
    ("f_quality",   "수익성 스프레드 (ROIC/ROE)"),
    ("f_mom",       "중기 모멘텀 (12-1개월)"),
    ("f_trend",     "200일선 추세 안정성"),
]

def get_val(frame: dict, col: str, y: int):
    v = (frame.get(str(y)) or {}).get(col)
    return None if v is None else float(v)

def cagr_avg(vals_early, vals_late, span: float, min_len: int = 2):
    if len(vals_early) < min_len or len(vals_late) < min_len:
        return None
    a, b = sum(vals_early) / len(vals_early), sum(vals_late) / len(vals_late)
    return (b / a) ** (1.0 / span) - 1.0 if (a > 0 and b > 0) else None

def get_pit_frame(comp: dict, fy: int) -> dict | None:
    pit_all = comp.get("pit") or {}
    if str(fy) in pit_all:
        return pit_all[str(fy)]
    if fy < 2018 and "2018" in pit_all:
        sliced = {yk: yv for yk, yv in pit_all["2018"].items() if int(yk) <= fy}
        return sliced if sliced else None
    return None

def extract_company_year(comp: dict, fy: int, rebal_dt: pd.Timestamp) -> dict | None:
    t = comp["ticker"]
    px_df = PRICE_MAP.get(t)
    if px_df is None or px_df.empty:
        return None
    hist = px_df.loc[:rebal_dt]
    if len(hist) < 120:
        return None
    px_now = float(hist["close"].iloc[-1])
    if px_now <= 0:
        return None
    frame = get_pit_frame(comp, fy)
    if not frame:
        return None
    path = comp.get("path") or "general"
    act_dps = {int(k): float(v) for k, v in (comp.get("act_dps") or {}).items() if v is not None}

    win_len = 3 if fy == 2016 else (4 if fy == 2017 else 5)
    dps_w = [get_val(frame, "dps_common", y) for y in range(fy - win_len + 1, fy + 1)]
    paid_w = sum(1 for v in dps_w if v is not None and v > 0)
    cuts_w = sum(1 for a, b in zip(dps_w, dps_w[1:]) if a is not None and b is not None and a > 0 and b < a * 0.99)
    if fy == 2016:
        g_dps = cagr_avg([v for y in (2014,) if (v := get_val(frame, "dps_common", y)) is not None], [v for y in (2016,) if (v := get_val(frame, "dps_common", y)) is not None], 2.0, 1)
        g_eps = cagr_avg([v for y in (2014,) if (v := get_val(frame, "eps_reported", y)) is not None], [v for y in (2016,) if (v := get_val(frame, "eps_reported", y)) is not None], 2.0, 1)
    elif fy == 2017:
        g_dps = cagr_avg([v for y in (2014, 2015) if (v := get_val(frame, "dps_common", y)) is not None], [v for y in (2016, 2017) if (v := get_val(frame, "dps_common", y)) is not None], 2.0, 2)
        g_eps = cagr_avg([v for y in (2014, 2015) if (v := get_val(frame, "eps_reported", y)) is not None], [v for y in (2016, 2017) if (v := get_val(frame, "eps_reported", y)) is not None], 2.0, 2)
    elif fy <= 2019:
        span = 3.0 if fy == 2018 else 3.5
        g_dps = cagr_avg([v for y in range(2013, 2016) if (v := get_val(frame, "dps_common", y)) is not None], [v for y in range(fy - 2, fy + 1) if (v := get_val(frame, "dps_common", y)) is not None], span)
        g_eps = cagr_avg([v for y in range(2013, 2016) if (v := get_val(frame, "eps_reported", y)) is not None], [v for y in range(fy - 2, fy + 1) if (v := get_val(frame, "eps_reported", y)) is not None], span)
    else:
        g_dps = cagr_avg([v for y in range(fy - 7, fy - 4) if (v := get_val(frame, "dps_common", y)) is not None], [v for y in range(fy - 2, fy + 1) if (v := get_val(frame, "dps_common", y)) is not None], 5.0)
        g_eps = cagr_avg([v for y in range(fy - 7, fy - 4) if (v := get_val(frame, "eps_reported", y)) is not None], [v for y in range(fy - 2, fy + 1) if (v := get_val(frame, "eps_reported", y)) is not None], 5.0)

    v1 = "FAIL" if (paid_w < max(2, win_len - 1) or cuts_w >= 2) else ("PASS" if (paid_w == win_len and cuts_w == 0 and g_dps is not None and g_dps > 0) else "WARN")

    d3 = [get_val(frame, "dividends_paid", y) if get_val(frame, "dividends_paid", y) is not None else get_val(frame, "div_total_reported", y) for y in range(fy - 2, fy + 1)]
    ni3 = [get_val(frame, "net_income_parent", y) if get_val(frame, "net_income_parent", y) is not None else get_val(frame, "ni_reported", y) for y in range(fy - 2, fy + 1)]
    tp3 = [get_val(frame, "treasury_purchase", y) or 0.0 for y in range(fy - 2, fy + 1)]
    canc3 = sum(get_val(frame, "ts_cancelled_common", y) or 0.0 for y in range(fy - 2, fy + 1))
    acq3 = sum(get_val(frame, "ts_acquired_common", y) or 0.0 for y in range(fy - 2, fy + 1))
    canc_ratio3 = min(1.0, canc3 / acq3) if acq3 > 0 else 0.0
    p3 = (sum(d3) / sum(ni3) * 100.0) if (all(v is not None for v in d3 + ni3) and sum(ni3) > 0) else None
    tr_3y = ((sum(d3) + sum(tp3)) / sum(ni3) * 100.0) if (all(v is not None for v in d3 + ni3) and sum(ni3) > 0) else None

    if g_eps is None or g_dps is None: v2 = "NA"
    elif g_eps < 0 and g_dps > 0: v2 = "FAIL"
    elif g_eps > 0 and (g_dps <= g_eps + 0.03 or (p3 is not None and p3 < 30.0)): v2 = "PASS"
    else: v2 = "WARN"

    if path == "general":
        k3 = [v for y in range(max(2014, fy - 4), fy + 1) if (v := get_val(frame, "fcf", y)) is not None]
        v3 = "FAIL" if sum(1 for v in k3 if v < 0) >= 2 else ("PASS" if (len(k3) >= 3 and sum(1 for v in k3 if v > 0) / len(k3) >= 0.75) else ("WARN" if len(k3) >= 2 else "NA"))
    else:
        eqs = [get_val(frame, "equity_parent", y) for y in range(max(2014, fy - 4), fy + 1)]
        eq_ch = [b - a for a, b in zip(eqs, eqs[1:]) if a is not None and b is not None]
        v3 = "FAIL" if sum(1 for v in eq_ch if v < 0) >= 2 else ("PASS" if (len(eq_ch) >= 2 and sum(1 for v in eq_ch if v > 0) / len(eq_ch) >= 0.75) else ("WARN" if eq_ch else "NA"))

    sps = []
    for y in range(max(2014, fy - 4), fy + 1):
        rf = RF_MAP.get(y, 0.025); coe = rf + 0.05
        if path != "general":
            roe = get_val(frame, "roe", y)
            if roe is not None: sps.append(roe - coe)
        else:
            roic = get_val(frame, "roic", y)
            if roic is not None:
                nd = get_val(frame, "net_debt", y)
                wacc = coe if (nd is None or nd <= 0) else (0.75 * coe + 0.25 * (rf + 0.015) * 0.78)
                sps.append(roic - wacc)
    avg5_sp = (sum(sps) / len(sps)) if sps else None
    p_thr = 0.03 if path == "general" else 0.01
    if len(sps) >= 3 and all(s < 0 for s in sps[-3:]): v4 = "FAIL"
    elif avg5_sp is not None: v4 = "PASS" if avg5_sp >= p_thr else "WARN"
    else: v4 = "NA"

    if tr_3y is None or p3 is None: v5 = "NA"
    elif tr_3y < 20.0 or p3 > 75.0 or (path != "general" and tr_3y > 100.0): v5 = "FAIL"
    elif tr_3y >= 30.0 and p3 <= 60.0 and (path == "general" or tr_3y <= 66.7): v5 = "PASS"
    else: v5 = "WARN"

    s_now = get_val(frame, "outstanding_common", fy)
    sh_old_y = fy - 5
    s_old = get_val(frame, "outstanding_common", sh_old_y)
    if not s_old:
        avail_sh = sorted(int(yk) for yk, yv in frame.items() if int(yk) <= fy - 3 and (yv.get("outstanding_common") or 0) > 0)
        if avail_sh:
            sh_old_y = min(avail_sh, key=lambda yk: abs((fy - yk) - 5))
            s_old = get_val(frame, "outstanding_common", sh_old_y)
    if s_now and s_old and s_old > 0 and fy > sh_old_y:
        sf_now = get_val(frame, "split_factor_to_latest", fy) or 1.0
        sf_old = get_val(frame, "split_factor_to_latest", sh_old_y) or 1.0
        sh_chg = ((s_now * sf_now) / (s_old * sf_old)) ** (5.0 / (fy - sh_old_y)) - 1.0
    else:
        sh_chg = None
    canc5 = sum(get_val(frame, "ts_cancelled_common", y) or 0.0 for y in range(max(2015, fy - 4), fy + 1))
    acq5 = sum(get_val(frame, "ts_acquired_common", y) or 0.0 for y in range(max(2015, fy - 4), fy + 1))
    if sh_chg is None: v6 = "NA"
    elif sh_chg >= 0.03: v6 = "FAIL"
    elif (sh_chg < -0.01 and canc5 > 0) or (-0.01 <= sh_chg <= 0.01 and (acq5 == 0 or canc5 >= acq5)): v6 = "PASS"
    else: v6 = "WARN"

    if path != "general":
        v7, v8 = "참고", "PASS"
    else:
        fcfs3 = [get_val(frame, "fcf", y) for y in range(fy - 2, fy + 1)]
        if any(v is None for v in d3 + fcfs3): v7 = "NA"
        elif sum(fcfs3) <= 0: v7 = "FAIL" if (sum(d3) + sum(tp3)) > 0 else "PASS"
        else:
            cov = (sum(d3) + sum(tp3)) / sum(fcfs3)
            v7 = "FAIL" if cov > 1.0 else ("WARN" if cov > 2 / 3 else "PASS")
        nd, nde, icr = get_val(frame, "net_debt", fy), get_val(frame, "net_debt_to_ebitda", fy), get_val(frame, "interest_coverage", fy)
        if nd is not None and nd <= 0: v8 = "PASS"
        elif (icr is not None and icr < 3.0) or (nde is not None and nde > 3.0): v8 = "FAIL"
        elif nde is not None and icr is not None: v8 = "PASS" if (nde < 2.0 and icr >= 5.0) else "WARN"
        else: v8 = "NA"

    counted = [v1, v2, v3, v4, v5, v6] + ([v7, v8] if path == "general" else [])
    n_fail, n_warn, n_na = counted.count("FAIL"), counted.count("WARN"), sum(1 for v in counted if v == "NA")
    pass_a = (n_fail == 0 and n_warn <= 2 and n_na <= 2)

    dps_m2, dps_m1, dps_0 = get_val(frame, "dps_common", fy - 2), get_val(frame, "dps_common", fy - 1), get_val(frame, "dps_common", fy)
    consec_2_cuts = bool(dps_m2 and dps_m1 is not None and dps_0 is not None and dps_m2 > 0 and dps_m1 < dps_m2 * 0.99 and dps_0 < dps_m1 * 0.99)
    sell_trigger = (n_fail > 0) or consec_2_cuts or (dps_0 is not None and dps_0 <= 0)

    dps_y_adj = act_dps.get(fy)
    dps_y_rep = get_val(frame, "dps_common", fy)
    adj_ratio = (dps_y_adj / dps_y_rep) if (dps_y_adj and dps_y_rep and dps_y_rep > 0) else 1.0
    eps_y, eq_y = get_val(frame, "eps_reported", fy), get_val(frame, "equity_parent", fy)
    sh_yr = fy
    sh_tot = (get_val(frame, "outstanding_common", fy) or 0) + (get_val(frame, "outstanding_pref", fy) or 0)
    if sh_tot <= 0 and fy < 2018:
        sh_yr = 2015
        sh_tot = (get_val(frame, "outstanding_common", 2015) or 0) + (get_val(frame, "outstanding_pref", 2015) or 0)
    split_fac = get_val(frame, "split_factor_to_latest", sh_yr)
    if split_fac is None or split_fac <= 0:
        split_fac = (1.0 / adj_ratio) if (adj_ratio and adj_ratio > 0) else 1.0
    pit_mcap_eok = (px_now * split_fac * sh_tot) / 1e8 if sh_tot > 0 else None

    bps_y = (eq_y / sh_tot) if (eq_y and sh_tot > 0) else None
    bps_adj_ratio = (1.0 / split_fac) if (sh_yr != fy and split_fac > 0) else adj_ratio
    c_per = px_now / (eps_y * adj_ratio) if (eps_y and eps_y > 0) else None
    c_pbr = px_now / (bps_y * bps_adj_ratio) if (bps_y and bps_y > 0) else None
    c_yld = dps_y_adj / px_now if (dps_y_adj and dps_y_adj > 0) else None

    h_per, h_pbr, h_yld = [], [], []
    for y in range(max(2014, fy - 7), fy + 1):
        sub_y = px_df.loc[f"{y}-01-01":f"{y}-12-31"]
        if sub_y.empty: continue
        py = float(sub_y["close"].iloc[-1])
        ey, eq = get_val(frame, "eps_reported", y), get_val(frame, "equity_parent", y)
        st = (get_val(frame, "outstanding_common", y) or 0) + (get_val(frame, "outstanding_pref", y) or 0)
        if st <= 0 and y < 2018:
            st = (get_val(frame, "outstanding_common", 2015) or 0) + (get_val(frame, "outstanding_pref", 2015) or 0)
        by = (eq / st) if (eq and st > 0) else None
        dy_a, dy_r = act_dps.get(y), get_val(frame, "dps_common", y)
        ar = (dy_a / dy_r) if (dy_a and dy_r and dy_r > 0) else adj_ratio
        if ey and ey > 0: h_per.append(py / (ey * ar))
        if by and by > 0: h_pbr.append(py / (by * ar))
        if dy_a and dy_a > 0: h_yld.append(dy_a / py)

    devs_raw, devs_cap = [], []
    cap_lim, min_hv = BLIND_CONFIG["VAL_CAP"], (2 if fy <= 2018 else 3)
    for cv, hv, hc in ((c_per, h_per, False), (c_pbr, h_pbr, False), (c_yld, h_yld, True)):
        if cv is not None and len(hv) >= min_hv:
            med = statistics.median(hv)
            if med > 0:
                d = (cv / med - 1.0) if hc else -(cv / med - 1.0)
                devs_raw.append(d); devs_cap.append(max(-cap_lim, min(cap_lim, d)))
    val_raw = sum(devs_raw) / len(devs_raw) if len(devs_raw) >= 2 else None
    val_cap = sum(devs_cap) / len(devs_cap) if len(devs_cap) >= 2 else None

    closes = hist["close"]
    mom_12_1 = float(closes.iloc[-21] / closes.iloc[-252] - 1.0) if len(closes) >= 252 else (float(closes.iloc[-21] / closes.iloc[0] - 1.0) if len(closes) >= 60 else 0.0)
    ma200 = float(closes.tail(200).mean())
    trend_200 = (px_now / ma200 - 1.0) if ma200 > 0 else 0.0
    dret = closes.tail(252).pct_change().dropna()
    ann_vol = float(dret.std() * np.sqrt(252)) if len(dret) >= 30 else 0.30
    turnover_60d_eok = float((hist["close"] * hist["volume"]).tail(60).mean()) / 1e8

    pit_mcap_won = (pit_mcap_eok * 1e8) if (pit_mcap_eok and pit_mcap_eok > 0) else None
    true_sh_yield = (((sum(d3) + sum(tp3) * canc_ratio3) / 3.0) / pit_mcap_won) if (pit_mcap_won and all(v is not None for v in d3)) else c_yld
    abs_over = (c_per is not None and c_per > 15.0) or (path == "general" and c_pbr is not None and c_pbr > 1.8)
    abs_ok = (not abs_over) and ((c_per is not None and c_pbr is not None) if path == "general" else True)

    return {
        "fy": fy, "rebal_dt": rebal_dt.strftime("%Y-%m-%d"),
        "ticker": t, "name": comp["name"], "is_fin": path != "general",
        "pass_a": pass_a, "sell_trigger": sell_trigger,
        "pit_mcap_eok": pit_mcap_eok, "turnover_60d_eok": turnover_60d_eok,
        "ann_vol": ann_vol, "per": c_per, "pbr": c_pbr, "abs_ok": abs_ok,
        "dps_next_adj": act_dps.get(fy + 1, act_dps.get(fy, 0.0)),
        "f_val_raw": val_raw,
        # 10개 후보 팩터
        "f_lowvol": -ann_vol,
        "f_inv_per": (-c_per if (c_per and c_per > 0) else np.nan),
        "f_inv_pbr": (-c_pbr if (c_pbr and c_pbr > 0) else np.nan),
        "f_yield": (c_yld if c_yld is not None else np.nan),
        "f_sh_yield": (true_sh_yield if true_sh_yield is not None else np.nan),
        "f_val_rel": (val_cap if val_cap is not None else np.nan),
        "f_dps_g": (g_dps if g_dps is not None else np.nan),
        "f_quality": (avg5_sp if avg5_sp is not None else np.nan),
        "f_mom": mom_12_1,
        "f_trend": trend_200,
    }

def build_panel_for_month(rebal_month: int = 4) -> tuple[pd.DataFrame, dict, dict]:
    rdates = {}
    for fy in range(BLIND_CONFIG["START_FY"], BLIND_CONFIG["END_FY"] + 1):
        sub = CALENDAR[CALENDAR >= pd.Timestamp(f"{fy + 1}-{rebal_month:02d}-01")]
        if len(sub) > 0:
            rdates[fy] = sub[0]
    sfys = sorted(rdates.keys())
    rows, sell_map = [], {}
    for idx_f, fy in enumerate(sfys):
        dt = rdates[fy]
        next_dt = rdates[sfys[idx_f + 1]] if idx_f + 1 < len(sfys) else CALENDAR[-1]
        sflags = {}
        for comp in COMPANIES:
            rec = extract_company_year(comp, fy, dt)
            if rec is None:
                continue
            sflags[rec["ticker"]] = rec["sell_trigger"]
            if (not rec["pass_a"]) or (rec["f_val_rel"] is np.nan) or pd.isna(rec["f_val_rel"]):
                continue
            if BLIND_CONFIG["PIT_MIN_MCAP_EOK"] > 0 and (rec["pit_mcap_eok"] is None or rec["pit_mcap_eok"] < BLIND_CONFIG["PIT_MIN_MCAP_EOK"]):
                continue
            sub_fwd = PRICE_MAP[rec["ticker"]].loc[dt:next_dt]
            if sub_fwd.empty:
                continue
            p0, p1 = float(sub_fwd["close"].iloc[0]), float(sub_fwd["close"].iloc[-1])
            rec["fwd_tr"] = (p1 + rec["dps_next_adj"] * (1.0 - BLIND_CONFIG["DIV_TAX_RATE"])) / p0 - 1.0
            rows.append(rec)
        sell_map[dt] = sflags
    return pd.DataFrame(rows), rdates, sell_map

PANEL_APR, RDATES_APR, SELL_MAP_APR = build_panel_for_month(4)
print(f"[패널 완료] 4월 기준 총 {len(PANEL_APR)}행 (탐색 FY16~20: {len(PANEL_APR[PANEL_APR['fy']<=BLIND_CONFIG['TRAIN_END_FY']])}행 | 봉인 OOS FY21~25: {len(PANEL_APR[PANEL_APR['fy']>=BLIND_CONFIG['OOS_START_FY']])}행)")

In [ ]:
# ==============================================================================
# 5. [백지상태 자동 탐색] 오직 탐색 구간(FY2016~FY2020, 2017-04~2022-04)만 열람!
# ==============================================================================
# ※ 절대 원칙: 아래 셀에서는 FY >= 2021 (2022-04 이후) 데이터를 한 줄도 읽지 않습니다!
TRAIN_PANEL = PANEL_APR[PANEL_APR["fy"] <= BLIND_CONFIG["TRAIN_END_FY"]].copy()
OOS_PANEL = PANEL_APR[PANEL_APR["fy"] >= BLIND_CONFIG["OOS_START_FY"]].copy()

def spearman_ic(x: pd.Series, y: pd.Series) -> float:
    df_xy = pd.DataFrame({"x": pd.to_numeric(x, errors="coerce"), "y": pd.to_numeric(y, errors="coerce")}).dropna()
    if len(df_xy) < 6:
        return np.nan
    rx, ry = df_xy["x"].rank(), df_xy["y"].rank()
    if rx.std() == 0 or ry.std() == 0:
        return np.nan
    return float(rx.corr(ry))

train_ic_rows = []
TRAIN_WEIGHTS_RAW = {}
for fcol, flab in CANDIDATE_FACTORS:
    ics = []
    for dt_str, grp in TRAIN_PANEL.groupby("rebal_dt"):
        g = grp[grp["turnover_60d_eok"] >= BLIND_CONFIG["MIN_LIQ_EOK"]]
        if len(g) < 8:
            g = grp
        ic = spearman_ic(g[fcol], g["fwd_tr"])
        if not np.isnan(ic):
            ics.append(ic)
    mic = float(np.mean(ics)) if ics else 0.0
    sic = float(np.std(ics, ddof=1)) if len(ics) > 1 else 1.0
    icir = mic / sic if sic > 0 else 0.0
    pos = sum(v > 0 for v in ics)
    TRAIN_WEIGHTS_RAW[fcol] = max(0.0, mic)
    train_ic_rows.append({"팩터": flab, "_col": fcol, "Train 평균IC": f"{mic:+.3f}", "Train ICIR": f"{icir:+.2f}", "양수연도": f"{pos}/{len(ics)}", "_icir": icir, "_mic": mic})

DF_TRAIN_IC = pd.DataFrame(train_ic_rows).sort_values("_icir", ascending=False).reset_index(drop=True)
BLIND_TOP3_COLS = DF_TRAIN_IC["_col"].head(3).tolist()
BLIND_TOP3_NAMES = DF_TRAIN_IC["팩터"].head(3).tolist()

# Blind-2: Train 양(+)의 IC 50% 수축 가중치 (2022-04 영구 동결!)
eq_w = 1.0 / len(CANDIDATE_FACTORS)
s_pos = sum(TRAIN_WEIGHTS_RAW.values())
BLIND_SHRUNK_W = {}
for fcol, _ in CANDIDATE_FACTORS:
    lw = (TRAIN_WEIGHTS_RAW[fcol] / s_pos) if s_pos > 0 else eq_w
    BLIND_SHRUNK_W[fcol] = 0.5 * lw + 0.5 * eq_w
sb = sum(BLIND_SHRUNK_W.values())
BLIND_SHRUNK_W = {k: v / sb for k, v in BLIND_SHRUNK_W.items()}

print("■ [백지상태 탐색 결과 — 오직 2017-04 ~ 2022-04 과거 5년만 열람]")
display(DF_TRAIN_IC[["팩터", "Train 평균IC", "Train ICIR", "양수연도"]])
print(f"\n  => Blind-1 자동 선택 Top-3 (2022-04 영구 동결): {BLIND_TOP3_NAMES}")
print("  => Blind-2 자동 수축 가중치 상위 5개:", {k: f"{v:.1%}" for k, v in sorted(BLIND_SHRUNK_W.items(), key=lambda x: -x[1])[:5]})

In [ ]:
# ==============================================================================
# 6. [봉인 해제 — Strict OOS 채점] 2022-04 ~ 최신 미열람 구간 성과 & 사후편향 거품 측정
# ==============================================================================
_PX_PANEL = pd.DataFrame({t: df["close"] for t, df in PRICE_MAP.items()}).reindex(CALENDAR).ffill()
_PX_ARR = np.nan_to_num(_PX_PANEL.to_numpy(dtype=np.float64), nan=0.0)
_TK_COL = {t: idx for idx, t in enumerate(_PX_PANEL.columns)}
_CAL_POS = {dt: idx for idx, dt in enumerate(CALENDAR)}
_ACT_DPS_MAP = {c["ticker"]: {int(k): float(v) for k, v in (c.get("act_dps") or {}).items() if v is not None} for c in COMPANIES}
_APRIL_SET = {CALENDAR[(CALENDAR.year == y) & (CALENDAR.month == 4)][0] for y in sorted(set(CALENDAR.year)) if len(CALENDAR[(CALENDAR.year == y) & (CALENDAR.month == 4)]) > 0}

def pct_rank(s: pd.Series, higher_better: bool = True) -> pd.Series:
    ns = pd.to_numeric(s, errors="coerce")
    return (ns if higher_better else -ns).rank(pct=True, method="average").fillna(0.5)

def build_weights(panel: pd.DataFrame, strategy: str, top_n: int = 10, max_fin_ratio: float | None = None):
    w_map, rk_map = {}, {}
    min_liq = BLIND_CONFIG["MIN_LIQ_EOK"]
    for dt_str in sorted(panel["rebal_dt"].unique()):
        dt = pd.Timestamp(dt_str)
        g_all = panel[panel["rebal_dt"] == dt_str].copy()
        g = g_all[g_all["turnover_60d_eok"] >= min_liq].copy()
        if len(g) < top_n:
            g = g_all.copy()

        if strategy == "Blind1_Top3":
            g["sc"] = sum(pct_rank(g[c], True) for c in BLIND_TOP3_COLS) / len(BLIND_TOP3_COLS)
        elif strategy == "Blind2_Shrunk":
            g["sc"] = sum(BLIND_SHRUNK_W[c] * pct_rank(g[c], True) for c, _ in CANDIDATE_FACTORS)
        elif strategy == "Blind3_Equal4":
            # 학습 0% 교과서 4팩터 균등 (저변동 + 절대PER + 진성주주환원 + 퀄리티)
            g["sc"] = 0.25 * (pct_rank(g["f_lowvol"]) + pct_rank(g["f_inv_per"]) + pct_rank(g["f_sh_yield"]) + pct_rank(g["f_quality"]))
        elif strategy == "PostHoc_Final2":
            # 비교용: 전체 기간을 본 뒤 고른 사후 최종안 ② (진성환원 + 상대70% 게이트 + 금융≤50%)
            r_vol = pct_rank(g["f_lowvol"], True)
            r_per = pct_rank(g["f_inv_per"], True)
            r_pbr = pct_rank(g["f_inv_pbr"].where(~g["is_fin"]), True)
            r_val = pct_rank(g["f_val_rel"].fillna(0.0), True)
            r_abs = np.where(g["is_fin"], 0.5 * r_per + 0.5 * r_val, 0.5 * r_per + 0.5 * r_pbr)
            bsc = 0.30 * r_vol + 0.30 * pd.Series(r_abs, index=g.index) + 0.20 * pct_rank(g["f_yield"]) + 0.20 * pct_rank(g["f_sh_yield"])
            vcut = float(g["ann_vol"].quantile(0.70))
            gate = g["abs_ok"] & (g["ann_vol"] <= vcut)
            g["sc"] = gate.astype(float) * 10.0 + bsc
        elif strategy == "StratA_Clean":
            g["sc"] = pct_rank(g["f_val_raw"], True) + pct_rank(g["f_dps_g"].fillna(0.0), True)
        elif strategy == "EW_Pass":
            w_map[dt] = {t: 1.0 / len(g) for t in g["ticker"]}
            rk_map[dt] = g["ticker"].tolist()
            continue
        else:
            raise ValueError(strategy)

        gs = g.sort_values("sc", ascending=False)
        if max_fin_ratio is not None:
            mfin = max(1, int(round(top_n * max_fin_ratio)))
            chosen, fc = [], 0
            for _, r in gs.iterrows():
                if len(chosen) >= top_n: break
                if r["is_fin"] and fc >= mfin: continue
                chosen.append(r)
                if r["is_fin"]: fc += 1
            if len(chosen) < top_n:
                used = {r["ticker"] for r in chosen}
                for _, r in gs.iterrows():
                    if r["ticker"] not in used:
                        chosen.append(r)
                        if len(chosen) >= top_n: break
            picks = pd.DataFrame(chosen)
            rk = picks["ticker"].tolist() + [t for t in gs["ticker"].tolist() if t not in set(picks["ticker"])]
        else:
            picks = gs.head(top_n)
            rk = gs["ticker"].tolist()

        w_map[dt] = {t: 1.0 / len(picks) for t in picks["ticker"]}
        rk_map[dt] = rk
    return w_map, rk_map

def run_lumpsum(weights_by_date: dict) -> pd.Series:
    rset = sorted(weights_by_date.keys())
    spos = _CAL_POS[rset[0]]
    cal = CALENDAR[spos:]
    nav = np.empty(len(cal), dtype=np.float64)
    fee, dtax = BLIND_CONFIG["FEE_RATE"], BLIND_CONFIG["DIV_TAX_RATE"]
    shares, cash, paid_y = {}, float(BLIND_CONFIG["INITIAL_CAPITAL"]), set()
    for i, dt in enumerate(cal):
        px = _PX_ARR[spos + i]
        if shares and (dt in _APRIL_SET) and (dt.year not in paid_y):
            gross = sum(sh * _ACT_DPS_MAP.get(tk, {}).get(dt.year - 1, 0.0) for tk, sh in shares.items())
            cash += gross * (1.0 - dtax)
            paid_y.add(dt.year)
            if (dt not in weights_by_date) and cash > 0:
                sv = sum(sh * px[_TK_COL[tk]] for tk, sh in shares.items() if tk in _TK_COL)
                if sv > 0:
                    inv = cash * (1.0 - fee)
                    for tk in list(shares.keys()):
                        p = px[_TK_COL[tk]]
                        if p > 0:
                            shares[tk] += (inv * (shares[tk] * p / sv)) / p
                    cash = 0.0
        if dt in weights_by_date:
            sv = sum(sh * px[_TK_COL[tk]] for tk, sh in shares.items() if tk in _TK_COL)
            tv = sv + cash
            vw = {tk: w for tk, w in weights_by_date[dt].items() if tk in _TK_COL and px[_TK_COL[tk]] > 0}
            ws = sum(vw.values())
            if ws > 0:
                vw = {tk: w / ws for tk, w in vw.items()}
                traded = sum(abs(tv * vw.get(tk, 0.0) - shares.get(tk, 0.0) * (px[_TK_COL[tk]] if tk in _TK_COL else 0.0)) for tk in (set(shares) | set(vw)))
                inv = max(0.0, tv - traded * fee)
                shares = {tk: (inv * w) / px[_TK_COL[tk]] for tk, w in vw.items()}
                cash = 0.0
        nav[i] = cash + sum(sh * px[_TK_COL[tk]] for tk, sh in shares.items() if tk in _TK_COL)
    return pd.Series(nav, index=cal)

def calc_xirr(cfs):
    t0 = cfs[0][0]
    cy = [((d - t0).days / 365.25, a) for d, a in cfs]
    lo, hi = -0.90, 5.0
    for _ in range(100):
        mid = 0.5 * (lo + hi)
        if sum(a / ((1.0 + mid) ** y) for y, a in cy) > 0: lo = mid
        else: hi = mid
    return 0.5 * (lo + hi)

def run_dca_oos(ranked_by_date: dict, sell_map: dict, top_n: int = 10) -> dict:
    rset = sorted(ranked_by_date.keys())
    spos = _CAL_POS[rset[0]]
    cal = CALENDAR[spos:]
    ym = cal.to_period("M")
    fm, seen = set(), set()
    for dt, m in zip(cal, ym):
        if m not in seen: fm.add(dt); seen.add(m)
    fee, dtax, mc = BLIND_CONFIG["FEE_RATE"], BLIND_CONFIG["DIV_TAX_RATE"], BLIND_CONFIG["MONTHLY_DCA_KRW"]
    shares, cash, paid_y, rk, cfs = {}, 0.0, set(), [], []
    for i, dt in enumerate(cal):
        px = _PX_ARR[spos + i]
        if shares and (dt in _APRIL_SET) and (dt.year not in paid_y):
            cash += sum(sh * _ACT_DPS_MAP.get(tk, {}).get(dt.year - 1, 0.0) for tk, sh in shares.items()) * (1.0 - dtax)
            paid_y.add(dt.year)
        if dt in ranked_by_date:
            rk = [tk for tk in ranked_by_date[dt] if tk in _TK_COL and px[_TK_COL[tk]] > 0]
            sf = sell_map.get(dt, {})
            for tk in list(shares.keys()):
                if sf.get(tk, False):
                    cash += shares.pop(tk) * px[_TK_COL[tk]] * (1.0 - fee)
        if dt in fm:
            cash += mc; cfs.append((dt, -mc))
        if cash > 0 and rk:
            tgts = [tk for tk in rk[:top_n] if tk in _TK_COL and px[_TK_COL[tk]] > 0]
            if tgts:
                per = cash * (1.0 - fee) / len(tgts)
                for tk in tgts:
                    shares[tk] = shares.get(tk, 0.0) + per / px[_TK_COL[tk]]
                cash = 0.0
    fv = cash + sum(sh * _PX_ARR[spos + len(cal) - 1, _TK_COL[tk]] for tk, sh in shares.items() if tk in _TK_COL)
    cfs.append((cal[-1], fv))
    return {"final_val": fv, "xirr": calc_xirr(cfs)}

def calc_perf(nav: pd.Series) -> dict:
    yrs = (nav.index[-1] - nav.index[0]).days / 365.25
    cagr = float((nav.iloc[-1] / nav.iloc[0]) ** (1.0 / yrs) - 1.0)
    mdd = float((nav / nav.cummax() - 1.0).min())
    vol = float(nav.pct_change().dropna().std() * np.sqrt(252))
    rf = float(np.mean([RF_MAP.get(y, 0.032) for y in sorted(set(nav.index.year))]))
    return {"cagr": cagr, "mdd": mdd, "vol": vol, "sharpe": (cagr - rf) / vol if vol > 0 else 0.0, "tot": float(nav.iloc[-1] / nav.iloc[0] - 1.0)}

OOS_STRATS = [
    ("Blind-1: 백지 자동 Top-3 균등 (2022 동결)",        "Blind1_Top3",   BLIND_CONFIG["MAX_FIN_RATIO"]),
    ("Blind-2: 백지 자동 IC수축 앙상블 (2022 동결)",     "Blind2_Shrunk", BLIND_CONFIG["MAX_FIN_RATIO"]),
    ("Blind-3: 학습 0% 교과서 4팩터 균등",              "Blind3_Equal4", BLIND_CONFIG["MAX_FIN_RATIO"]),
    ("비교(사후): 최종안 ② (전체열람 후 선정)",          "PostHoc_Final2",BLIND_CONFIG["MAX_FIN_RATIO"]),
    ("기준선: 현행 전략 A (클린, 유동성10억)",           "StratA_Clean",  None),
    ("기준선: 원칙부합 전체 동일가중 (EW)",              "EW_Pass",       None),
]

OOS_NAVS, OOS_METRICS, OOS_ROWS = {}, {}, []
oos_st = pd.Timestamp(sorted(OOS_PANEL["rebal_dt"].unique())[0])
k_oos = KOSPI_DF.loc[oos_st:, "close"]
KOSPI_TR_OOS = (k_oos / k_oos.iloc[0] * BLIND_CONFIG["INITIAL_CAPITAL"]) * ((1.0 + BLIND_CONFIG["KOSPI_EST_DIV_YIELD"]) ** ((k_oos.index - k_oos.index[0]).days / 365.25))

for lbl, code, fcap in OOS_STRATS:
    w_o, rk_o = build_weights(OOS_PANEL, code, BLIND_CONFIG["TOP_N"], fcap)
    nav_o = run_lumpsum(w_o)
    OOS_NAVS[lbl] = nav_o
    m = calc_perf(nav_o)
    OOS_METRICS[code] = m
    dca_o = run_dca_oos(rk_o, SELL_MAP_APR, BLIND_CONFIG["TOP_N"]) if code != "EW_Pass" else None
    OOS_ROWS.append({
        "전략 (완전 봉인 OOS: 2022-04 ~ 최신)": lbl,
        "OOS CAGR": f"{m['cagr']:+.2%}",
        "OOS MDD": f"{m['mdd']:.1%}",
        "OOS 샤프": f"{m['sharpe']:.2f}",
        "OOS 적립 XIRR": "-" if dca_o is None else f"{dca_o['xirr']:+.2%}",
    })

mk_o = calc_perf(KOSPI_TR_OOS)
OOS_ROWS.append({"전략 (완전 봉인 OOS: 2022-04 ~ 최신)": "KOSPI TR 추정 (+1.8%/년)", "OOS CAGR": f"{mk_o['cagr']:+.2%}", "OOS MDD": f"{mk_o['mdd']:.1%}", "OOS 샤프": f"{mk_o['sharpe']:.2f}", "OOS 적립 XIRR": "-"})
DF_OOS = pd.DataFrame(OOS_ROWS)

# OOS 무작위 500회 몬테카를로
pool_oos = {pd.Timestamp(dt_str): grp[grp["turnover_60d_eok"] >= BLIND_CONFIG["MIN_LIQ_EOK"]]["ticker"].tolist() for dt_str, grp in OOS_PANEL.groupby("rebal_dt")}
mc_cagrs, mc_sharpes = [], []
for i in range(BLIND_CONFIG["MC_RUNS"]):
    rng = random.Random(BLIND_CONFIG["MC_SEED"] + i)
    wr = {dt: {tk: 1.0 / min(BLIND_CONFIG["TOP_N"], len(tks)) for tk in rng.sample(tks, min(BLIND_CONFIG["TOP_N"], len(tks)))} for dt, tks in pool_oos.items()}
    mr = calc_perf(run_lumpsum(wr))
    mc_cagrs.append(mr["cagr"]); mc_sharpes.append(mr["sharpe"])
MC_CAGRS = np.array(mc_cagrs); MC_SHARPES = np.array(mc_sharpes)

best_blind_code = max(["Blind1_Top3", "Blind2_Shrunk", "Blind3_Equal4"], key=lambda k: OOS_METRICS[k]["sharpe"])
hindsight_gap = OOS_METRICS["PostHoc_Final2"]["cagr"] - OOS_METRICS[best_blind_code]["cagr"]
p_blind_cagr = float(np.mean(MC_CAGRS >= OOS_METRICS[best_blind_code]["cagr"]))
p_blind_sharpe = float(np.mean(MC_SHARPES >= OOS_METRICS[best_blind_code]["sharpe"]))
p_post_cagr = float(np.mean(MC_CAGRS >= OOS_METRICS["PostHoc_Final2"]["cagr"]))

print("■ [완전 봉인 OOS 채점표: 2022-04 ~ 최신 (4.7년, 학습 정보 0%)]")
display(DF_OOS)
print(f"\n■ [사후 선별 편향(Hindsight Gap) 진단]")
print(f"  - 최우수 백지 전략({best_blind_code}) OOS CAGR: {OOS_METRICS[best_blind_code]['cagr']:+.2%} (샤프 {OOS_METRICS[best_blind_code]['sharpe']:.2f}, 무작위 500회 중 상위 {p_blind_sharpe:.1%})")
print(f"  - 사후 최종안 ② OOS CAGR:                      {OOS_METRICS['PostHoc_Final2']['cagr']:+.2%} (샤프 {OOS_METRICS['PostHoc_Final2']['sharpe']:.2f}, 무작위 상위 {p_post_cagr:.1%})")
print(f"  - 추정 사후 편향 거품 (PostHoc - Best Blind):  {hindsight_gap*100:+.2f}%p/년  |  OOS 무작위 중앙값 CAGR: {np.median(MC_CAGRS):+.2%}")

In [ ]:
# ==============================================================================
# 7. [OOS 견고성 스트레스 테스트] 종목 수(8/10/15/20) & 리밸런싱 월 이동(4월/5월/10월)
# ==============================================================================
# (A) 종목 수 민감도 (OOS 2022-04 ~ 최신)
sens_rows = []
for n_test in BLIND_CONFIG["SENS_TOP_NS"]:
    row = {"보유 종목 수": f"{n_test}종목"}
    for col_nm, code, fcap in [
        ("현행 A", "StratA_Clean", None),
        ("Blind-1(Top3)", "Blind1_Top3", BLIND_CONFIG["MAX_FIN_RATIO"]),
        ("Blind-2(수축)", "Blind2_Shrunk", BLIND_CONFIG["MAX_FIN_RATIO"]),
        ("Blind-3(균등4)", "Blind3_Equal4", BLIND_CONFIG["MAX_FIN_RATIO"]),
        ("사후 최종안②", "PostHoc_Final2", BLIND_CONFIG["MAX_FIN_RATIO"]),
    ]:
        w_t, _ = build_weights(OOS_PANEL, code, n_test, fcap)
        mt = calc_perf(run_lumpsum(w_t))
        row[col_nm] = f"{mt['cagr']:+.1%} (Sh {mt['sharpe']:.2f})"
    sens_rows.append(row)
DF_SENS_N = pd.DataFrame(sens_rows)

# (B) 리밸런싱 시점 이동 테스트 (4월 초 vs 5월 초 vs 10월 초)
shift_rows = []
for mo in BLIND_CONFIG["SHIFT_MONTHS"]:
    p_m, _, _ = (PANEL_APR, RDATES_APR, SELL_MAP_APR) if mo == 4 else build_panel_for_month(mo)
    oos_m = p_m[p_m["fy"] >= BLIND_CONFIG["OOS_START_FY"]]
    row = {"리밸런싱 시점": f"매년 {mo}월 초"}
    for col_nm, code, fcap in [
        ("현행 A", "StratA_Clean", None),
        ("Blind-1(Top3)", "Blind1_Top3", BLIND_CONFIG["MAX_FIN_RATIO"]),
        ("Blind-2(수축)", "Blind2_Shrunk", BLIND_CONFIG["MAX_FIN_RATIO"]),
        ("Blind-3(균등4)", "Blind3_Equal4", BLIND_CONFIG["MAX_FIN_RATIO"]),
        ("사후 최종안②", "PostHoc_Final2", BLIND_CONFIG["MAX_FIN_RATIO"]),
    ]:
        w_m, _ = build_weights(oos_m, code, BLIND_CONFIG["TOP_N"], fcap)
        mm = calc_perf(run_lumpsum(w_m))
        row[col_nm] = f"{mm['cagr']:+.1%} (Sh {mm['sharpe']:.2f})"
    shift_rows.append(row)
DF_SHIFT_M = pd.DataFrame(shift_rows)

print("■ [OOS 견고성 ①] 종목 수 민감도 (8 · 10 · 15 · 20종목)")
display(DF_SENS_N)
print("\n■ [OOS 견고성 ②] 리밸런싱 월 이동 테스트 (4월 초 vs 5월 초 vs 10월 초)")
display(DF_SHIFT_M)

In [ ]:
# ==============================================================================
# 8. 4패널 블라인드 OOS 대시보드 + A4 PDF/HTML 리포트 Drive 자동 저장
# ==============================================================================
def export_html_to_pdf(html_str: str, pdf_path: str, chart_b64: str) -> bool:
    try:
        from weasyprint import HTML as WP_HTML
        WP_HTML(string=html_str).write_pdf(pdf_path)
        return True
    except Exception:
        pass
    try:
        from matplotlib.backends.backend_pdf import PdfPages
        import matplotlib.image as mpimg
        with PdfPages(pdf_path) as pdf:
            img = mpimg.imread(io.BytesIO(base64.b64decode(chart_b64)), format="png")
            fig_p = plt.figure(figsize=(11.69, 8.27))
            plt.imshow(img); plt.axis("off"); plt.tight_layout(); pdf.savefig(fig_p, dpi=150); plt.close(fig_p)
        return True
    except Exception:
        return False

as_of_str = datetime.now(KST).strftime("%Y-%m-%d")
run_stamp = datetime.now(KST).strftime("%Y-%m-%d_%H%M%S")

fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15.6, 8.9), facecolor="white")
fig.suptitle("진성 블라인드(백지상태) Out-of-Sample 검증 대시보드 (탐색 17~22 -> 봉인 22~26)", fontsize=15.0, fontweight="bold", color="#0F2922", y=0.985)
fig.text(0.5, 0.946, f"{DATASET_LABEL}  |  탐색(Train) 2017~2022만 보고 2022-04 영구 동결  |  사후 편향 거품 = {hindsight_gap*100:+.2f}%p/년", ha="center", fontsize=9.2, color="#64748B")

# (1) OOS Cumulative Return Curves
pal = [
    ("Blind-1: 백지 자동 Top-3 균등 (2022 동결)", "#059669", 2.4, "-"),
    ("Blind-2: 백지 자동 IC수축 앙상블 (2022 동결)", "#0F766E", 2.0, "-"),
    ("Blind-3: 학습 0% 교과서 4팩터 균등", "#D97706", 1.8, "-"),
    ("비교(사후): 최종안 ② (전체열람 후 선정)", "#7C3AED", 2.2, "--"),
    ("기준선: 현행 전략 A (클린, 유동성10억)", "#1D4ED8", 1.7, "-"),
]
for k, col, lw, ls in pal:
    s = (OOS_NAVS[k] / OOS_NAVS[k].iloc[0] - 1.0) * 100
    ax1.plot(s.index, s, label=f"{k.split(':')[0]} ({s.iloc[-1]:+.0f}%)", color=col, lw=lw, ls=ls)
ks = (KOSPI_TR_OOS / KOSPI_TR_OOS.iloc[0] - 1.0) * 100
ax1.plot(ks.index, ks, label=f"KOSPI TR ({ks.iloc[-1]:+.0f}%)", color="#64748B", lw=1.5, ls=":")
ax1.set_title("① 완전 봉인 OOS 누적 수익률 (2022-04 ~ 최신)", fontsize=11, fontweight="bold", color="#0F2922", loc="left")
ax1.yaxis.set_major_formatter(mtick.PercentFormatter())
ax1.legend(loc="upper left", fontsize=7.8, frameon=True, facecolor="white", ncol=2)
ax1.grid(True, color="#E2E8F0", ls="--", lw=0.7); ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False)

# (2) Train-Only (17~22) Factor ICIR bar chart
tp = DF_TRAIN_IC.iloc[::-1]
cols_b = ["#059669" if c in BLIND_TOP3_COLS else ("#38BDF8" if v > 0 else "#EF4444") for c, v in zip(tp["_col"], tp["_icir"])]
ax2.barh(tp["팩터"], tp["_icir"], color=cols_b, height=0.6)
ax2.axvline(0, color="#334155", lw=0.9)
ax2.set_title("② 탐색 구간(2017~2022)만 보고 계산한 팩터 ICIR (녹색=자동 Top3)", fontsize=11, fontweight="bold", color="#0F2922", loc="left")
ax2.grid(True, axis="x", color="#E2E8F0", ls="--", lw=0.7); ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False)

# (3) OOS Monte Carlo 500x Histogram
ax3.hist(MC_CAGRS * 100, bins=25, color="#CBD5E1", edgecolor="white", label=f"OOS 무작위 10종목 {BLIND_CONFIG['MC_RUNS']}회")
ax3.axvline(OOS_METRICS["StratA_Clean"]["cagr"] * 100, color="#1D4ED8", lw=2.0, ls="--", label=f"현행 A ({OOS_METRICS['StratA_Clean']['cagr']:+.1%})")
ax3.axvline(OOS_METRICS[best_blind_code]["cagr"] * 100, color="#059669", lw=2.5, label=f"최우수 Blind ({OOS_METRICS[best_blind_code]['cagr']:+.1%}, 상위 {p_blind_cagr:.1%})")
ax3.axvline(OOS_METRICS["PostHoc_Final2"]["cagr"] * 100, color="#7C3AED", lw=2.2, ls=":", label=f"사후 최종안② ({OOS_METRICS['PostHoc_Final2']['cagr']:+.1%})")
ax3.set_title("③ OOS 무작위 500회 대비 통계적 위치 (CAGR)", fontsize=11, fontweight="bold", color="#0F2922", loc="left")
ax3.set_xlabel("OOS 연평균 수익률 (CAGR, %)"); ax3.set_ylabel("빈도")
ax3.legend(loc="upper left", fontsize=8.0, frameon=True, facecolor="white")
ax3.grid(True, color="#E2E8F0", ls="--", lw=0.7); ax3.spines["top"].set_visible(False); ax3.spines["right"].set_visible(False)

# (4) OOS Sharpe Comparison Bar
sh_items = [
    ("현행 전략 A", OOS_METRICS["StratA_Clean"]["sharpe"], "#1D4ED8"),
    ("동일가중(EW)", OOS_METRICS["EW_Pass"]["sharpe"], "#94A3B8"),
    ("Blind-1\n(자동Top3)", OOS_METRICS["Blind1_Top3"]["sharpe"], "#059669"),
    ("Blind-2\n(IC수축)", OOS_METRICS["Blind2_Shrunk"]["sharpe"], "#0F766E"),
    ("Blind-3\n(교과서균등)", OOS_METRICS["Blind3_Equal4"]["sharpe"], "#D97706"),
    ("사후 최종안②", OOS_METRICS["PostHoc_Final2"]["sharpe"], "#7C3AED"),
    ("KOSPI TR", mk_o["sharpe"], "#64748B"),
]
xs = [x[0] for x in sh_items]; ys = [x[1] for x in sh_items]; cs = [x[2] for x in sh_items]
bars4 = ax4.bar(xs, ys, color=cs, width=0.58)
for b, y in zip(bars4, ys):
    ax4.text(b.get_x() + b.get_width() / 2, y + 0.02, f"{y:.2f}", ha="center", va="bottom", fontsize=8.5, fontweight="bold")
ax4.set_title("④ 완전 봉인 OOS 샤프지수 비교 (높을수록 우수)", fontsize=11, fontweight="bold", color="#0F2922", loc="left")
ax4.set_ylim(0, max(ys) + 0.15)
ax4.grid(True, axis="y", color="#E2E8F0", ls="--", lw=0.7); ax4.spines["top"].set_visible(False); ax4.spines["right"].set_visible(False)

fig.subplots_adjust(top=0.90, bottom=0.08, left=0.07, right=0.98, hspace=0.34, wspace=0.22)
buf = io.BytesIO()
fig.savefig(buf, format="png", dpi=150, bbox_inches="tight", facecolor="white")
DASH_B64 = base64.b64encode(buf.getvalue()).decode()
if IN_COLAB:
    plt.show()
plt.close(fig)

def render_tbl(df: pd.DataFrame) -> str:
    ths = "".join(f"<th>{escape(str(c))}</th>" for c in df.columns)
    trs = ["<tr>" + "".join(f'<td class="{"col-left" if i==0 else ""}">{escape(str(r[c]))}</td>' for i, c in enumerate(df.columns)) + "</tr>" for _, r in df.iterrows()]
    return f'<table class="tbl"><thead><tr>{ths}</tr></thead><tbody>{"".join(trs)}</tbody></table>'

html_report = f"""<!doctype html>
<html lang="ko"><head><meta charset="utf-8"><title>진성 블라인드 OOS 검증 리포트 ({as_of_str})</title>
<style>
  @page {{ size: A4 landscape; margin: 10mm; }}
  body {{ margin:0; padding:12px; background:#F8FAFC; color:#0F172A; font-family:'NanumGothic',sans-serif; font-size:10.5px; }}
  .wrap {{ max-width:1120px; margin:0 auto; }}
  .head {{ background:linear-gradient(135deg,#0F2922,#134E4A); color:#fff; padding:16px 20px; border-radius:10px; margin-bottom:12px; border-bottom:4px solid #D4A34A; }}
  .card {{ background:#fff; border:1px solid #E2E8F0; border-radius:10px; padding:12px 14px; margin-bottom:10px; break-inside:avoid; }}
  table.tbl {{ width:100%; border-collapse:collapse; font-size:9.3px; }}
  table.tbl th {{ background:#F1F5F9; padding:5px; border-top:1.5px solid #0F2922; border-bottom:1px solid #CBD5E1; text-align:right; }}
  table.tbl td {{ padding:5px; border-bottom:1px solid #EEF2F6; text-align:right; }}
  table.tbl td.col-left, table.tbl th:first-child {{ text-align:left; font-weight:700; }}
  img.chart {{ width:100%; height:auto; display:block; }}
  .page-break {{ page-break-before:always; break-before:page; }}
</style></head><body><div class="wrap">
  <div class="head">
    <h1 style="margin:0 0 4px;font-size:18px;">진성 블라인드(백지상태) Out-of-Sample 검증 리포트 ({as_of_str})</h1>
    <div>탐색(Train) 2017~2022만 열람 후 동결 -> 봉인(OOS) 2022~2026 단독 채점 | 추정 사후 편향 거품: {hindsight_gap*100:+.2f}%p/년</div>
  </div>
  <div class="card"><h2 style="margin:0 0 8px;">01. 4패널 블라인드 OOS 대시보드</h2><img class="chart" src="data:image/png;base64,{DASH_B64}"></div>
  <div class="card page-break"><h2 style="margin:0 0 8px;">02. 완전 봉인 OOS 채점표 (2022-04 ~ 최신)</h2>{render_tbl(DF_OOS)}</div>
  <div class="card"><h2 style="margin:0 0 8px;">03. 탐색 구간(2017~2022)만 본 10개 팩터 ICIR</h2>{render_tbl(DF_TRAIN_IC[['팩터','Train 평균IC','Train ICIR','양수연도']])}</div>
  <div class="card page-break"><h2 style="margin:0 0 8px;">04. OOS 종목 수 민감도 (8 · 10 · 15 · 20종목)</h2>{render_tbl(DF_SENS_N)}</div>
  <div class="card"><h2 style="margin:0 0 8px;">05. OOS 리밸런싱 월 이동 테스트 (4월 · 5월 · 10월)</h2>{render_tbl(DF_SHIFT_M)}</div>
</div></body></html>"""

r_html = f"blind_oos_report_{run_stamp}.html"
r_pdf = f"blind_oos_report_{run_stamp}.pdf"
Path(r_html).write_text(html_report, encoding="utf-8")
pdf_ok = export_html_to_pdf(html_report, r_pdf, DASH_B64)
print(f"[블라인드 리포트 완료] HTML={Path(r_html).stat().st_size/1024:.1f} KB | PDF={'성공' if pdf_ok else '건너뜀'}")

if DRIVE_REPORT_DIR is not None:
    try:
        (DRIVE_REPORT_DIR / r_html).write_text(html_report, encoding="utf-8")
        if pdf_ok and Path(r_pdf).exists():
            (DRIVE_REPORT_DIR / r_pdf).write_bytes(Path(r_pdf).read_bytes())
        DF_OOS.to_csv(DRIVE_REPORT_DIR / f"blind_oos_summary_{run_stamp}.csv", index=False, encoding="utf-8-sig")
        print(f"[Drive 저장 완료] {DRIVE_REPORT_DIR / r_pdf}")
    except Exception as e:
        print(f"[주의] Drive 저장 실패({type(e).__name__}): {e}")